In [1]:
!pip install portalocker decord deepspeed accelerate einops matplotlib numpy opencv-python pandas rich rouge sacrebleu scikit-image scikit-learn scipy seaborn tensorboard tensorflow timm tokenizers tqdm transformers sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.4 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 76.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.5 MB/s eta 

In [5]:
output_dir="out/stage1_pretraining"
!deepspeed --include localhost:0,1 --master_port 29511 /kaggle/input/model03/pytorch/default/12/Model01/pre_training.py \
  --batch-size 6 \
  --gradient-accumulation-steps 6 \
  --epochs 3\
  --opt AdamW \
  --lr 4e-4 \
  --quick_break 2048 \
  --output_dir out/stage1_pretraining \
  --dataset MS-ASL 

[2025-06-08 13:44:27,667] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
2025-06-08 13:44:33.152079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749390273.176504     683 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749390273.184062     683 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[2025-06-08 13:44:35,672] [WARNING] [runner.py:215:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2025-06-08 13:44:35,673] [INFO] [runner.py:605:main] cmd = /usr/bin/python3 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.

In [ ]:
output_dir= "out/stage2_pretraining"
ckpt_path= "/kaggle/working/out/stage1_pretraining/best_checkpoint.pth"
!deepspeed --include localhost:0,1 --master_port 29511 /kaggle/input/model03/pytorch/default/13/Model01/pre_training.py \
   --batch-size 5 \
   --gradient-accumulation-steps 5 \
   --epochs 5 \
   --opt AdamW \
   --lr 3e-4 \
   --quick_break 2048 \
   --output_dir $output_dir \
   --finetune $ckpt_path \
   --dataset MS-ASL \
   --rgb_support

In [13]:
output_dir="out/stage3_finetuning"

# RGB-pose setting
ckpt_path="/kaggle/working/out/stage2_pretraining/best_checkpoint.pth"

!deepspeed --include localhost:0,1 --master_port 29511 /kaggle/input/model03/pytorch/default/14/Model01/fine_tuning.py \
  --batch-size 8 \
  --gradient-accumulation-steps 1 \
  --epochs 20 \
  --opt AdamW \
  --lr 3e-4 \
  --output_dir $output_dir \
  --finetune $ckpt_path \
  --dataset MS-ASL \
  --task SLT \
  --rgb_support # enable RGB-pose setting

# example of ISLR
# deepspeed --include localhost:0,1,2,3 --master_port 29511 fine_tuning.py \
#    --batch-size 8 \
#    --gradient-accumulation-steps 1 \
#    --epochs 20 \
#    --opt AdamW \
#    --lr 3e-4 \
#    --output_dir $output_dir \
#    --finetune $ckpt_path \
#    --dataset WLASL \
#    --task ISLR \
#    --max_length 64 \
#    --rgb_support # enable RGB-pose setting

# # pose only setting
# ckpt_path=out/stage1_pretraining/final_checkpoint.pth

# deepspeed --include localhost:0,1 --master_port 29511 /kaggle/input/model03/pytorch/default/11/Model01/fine_tuning.py \
#  --batch-size 8 \
#  --gradient-accumulation-steps 1 \
#  --epochs 20 \
#  --opt AdamW \
#  --lr 3e-4 \
#  --output_dir $output_dir \
#  --finetune $ckpt_path \
#  --dataset MS-ASL \
#  --task SLT \
# #   --rgb_support # enable RGB-pose setting

[2025-06-07 05:27:33,701] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
2025-06-07 05:27:38.676357: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749274058.701270   13424 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749274058.710546   13424 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[2025-06-07 05:27:41,314] [WARNING] [runner.py:215:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2025-06-07 05:27:41,314] [INFO] [runner.py:605:main] cmd = /usr/bin/python3 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.

In [ ]:
ckpt_path="/kaggle/working/out/stage3_finetuning/best_checkpoint.pth"

single gpu inference
RGB-pose setting
deepspeed --include localhost:0 --master_port 29511 /kaggle/input/model03/pytorch/default/13/Model01/fine_tuning.py \
   --batch-size 8 \
   --gradient-accumulation-steps 1 \
   --epochs 20 \
   --opt AdamW \
   --lr 3e-4 \
   --output_dir out/test \
   --finetune $ckpt_path \
   --dataset CSL_Daily \
   --task SLT \
   --eval \
   --rgb_support

# example of ISLR
# deepspeed --include localhost:0,1,2,3 --master_port 29511 fine_tuning.py \
#    --batch-size 8 \
#    --gradient-accumulation-steps 1 \
#    --epochs 20 \
#    --opt AdamW \
#    --lr 3e-4 \
#    --output_dir $output_dir \
#    --finetune $ckpt_path \
#    --dataset WLASL \
#    --task ISLR \
#    --max_length 64 \
#    --rgb_support # enable RGB-pose setting

# # pose only setting
# ckpt_path=out/stage1_pretraining/final_checkpoint.pth

# deepspeed --include localhost:0,1 --master_port 29511 /kaggle/input/model03/pytorch/default/10/Model01/fine_tuning.py \
#  --batch-size 8 \
#  --gradient-accumulation-steps 1 \
#  --epochs 20 \
#  --opt AdamW \
#  --lr 3e-4 \
#  --output_dir $output_dir \
#  --finetune $ckpt_path \
#  --dataset CSL_Daily \
#  --task SLT \
# #   --rgb_support # enable RGB-pose setting


In [ ]:
# import pickle
# import numpy as np

# pkl_file_path = "/kaggle/input/ms-asl/MS-ASL/pose_format/100_test_2614.pkl"  # Update with actual path

# with open(pkl_file_path, 'rb') as f:
#     data = pickle.load(f)

# if isinstance(data, dict):
#     keypoints = data.get('keypoints')
#     if keypoints:
#         print("Keypoints type:", type(keypoints))
#         print("Keypoints length:", len(keypoints))
#         print("Shape of first keypoints array:", keypoints[0].shape)
#         print("Dimensions of first keypoints array:", keypoints[0].ndim)
#         print("First few rows:", keypoints[0][:5])